In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 10 — FUNCTION 4 (TRANSPARENCY + INTERPRETABILITY v5)
# Goals vs Week 9:
#  - Enforce DOMAIN BOUNDS explicitly: x in [0,1]^4 (raw space)
#  - More transparent decision log: why x_next was chosen + what drove it
#  - Reproducibility: fixed seeds, fewer moving parts, logged config + metrics
#  - Interpretability add-ons:
#       (a) nearest-neighbour context (closest observed points + y)
#       (b) local sensitivity (gradient of surrogate mean wrt inputs at x_next)
#       (c) uncertainty + distance penalty reported explicitly
#  - Still deterministic: ALWAYS outputs best-by-score non-duplicate
#  - x_next printed to 6 decimals or less
# ============================================================

# ----------------------------
# 0) Helpers
# ----------------------------
def clamp01(a):
    return np.minimum(1.0, np.maximum(0.0, a))

def fmt_x6(x):
    return "[" + ", ".join(f"{float(v):.6f}" for v in x) + "]"

def is_duplicate(x, X_existing, tol=1e-6):
    return np.any(np.linalg.norm(X_existing - x, axis=1) < tol)

def nearest_dist_scaled(cands_scaled, X_scaled_existing):
    diff = cands_scaled[:, None, :] - X_scaled_existing[None, :, :]
    d2 = np.sum(diff * diff, axis=2)
    return np.sqrt(np.min(d2, axis=1) + 1e-12)

def topk_nearest(x, X, y, k=3):
    d = np.linalg.norm(X - x[None, :], axis=1)
    idx = np.argsort(d)[:k]
    return idx, d[idx], y[idx]

# ----------------------------
# 1) Input data (Function 4) — RAW
# ----------------------------
X_train_raw = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
    [0.8893564 , 0.49958786, 0.53926886, 0.50878344],
    [0.25094624, 0.03369313, 0.14538002, 0.49493242],
    [0.34696206, 0.0062504 , 0.76056361, 0.61302356],
    [0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
    [0.80130271, 0.50023109, 0.70664456, 0.19510284],
    [0.24770826, 0.06044543, 0.04218635, 0.44132425],
    [0.74670224, 0.7570915 , 0.36935306, 0.20656628],
    [0.40066503, 0.07257425, 0.88676825, 0.24384229],
    [0.6260706 , 0.58675126, 0.43880578, 0.77885769],
    [0.95713529, 0.59764438, 0.76611385, 0.77620991],
    [0.73281243, 0.14524998, 0.47681272, 0.13336573],
    [0.65511548, 0.07239183, 0.68715175, 0.08151656],
    [0.21973443, 0.83203134, 0.48286416, 0.08256923],
    [0.48859419, 0.2119651 , 0.93917791, 0.37619173],
    [0.16713049, 0.87655456, 0.21723954, 0.95980098],
    [0.21691119, 0.16608583, 0.24137226, 0.77006248],
    [0.38748784, 0.80453226, 0.75179548, 0.72382744],
    [0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
    [0.03782483, 0.66485335, 0.16198218, 0.25392378],
    [0.68348638, 0.9027701 , 0.33541983, 0.99948256],
    [0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
    [0.85965692, 0.91959232, 0.20613873, 0.09779683],
    [0.28213837, 0.50598691, 0.53053084, 0.09630162],
    [0.32607578, 0.4723669 , 0.453192  , 0.10588734],
    [0.94838936, 0.89451301, 0.85163782, 0.55219629],
    [0.66495539, 0.04656628, 0.11677747, 0.79371778],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.73861301, 0.48210263, 0.70936644, 0.50397001],
    [0.8548108 , 0.49396462, 0.73530997, 0.80809201],
    [1.085621  , 1.019592  , 1.039177  , 1.099482  ],   # out-of-bounds in raw
    [1.00000e-06, 1.00000e-06, 1.24558e-01, 1.00000e-06],
    [0.866175  , 0.601115  , 0.708072  , 0.020585  ],
    [0.145904  , 0.536548  , 0.6014    , 0.01905   ],
    [0.356293  , 0.442523  , 0.13052   , 0.242559  ],
    [0.061431  , 0.381247  , 0.983792  , 0.705575  ],
    [0.497045, 0.450388, 0.380113, 0.297612],
    [0.480706, 0.444032, 0.354963, 0.354729],
    [0.503198, 0.435617, 0.371338, 0.408316]
], dtype=float)

y_train = np.array([
    -22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
    -15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
    -27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
    -26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
    -29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
    -32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284 ,
    -67.60493430274798, -22.782193418373407, -22.194212794446454,
    -13.363105653346768,  -5.926020577803715, -23.786955955997737,
    -2.5615259470796796, -0.81371612670717, -1.1642621740684471
], dtype=float)

assert len(X_train_raw) == len(y_train), "X_train and y_train length mismatch"

# ----------------------------
# 2) Enforce domain bounds [0,1] for training + duplicates + search
#    (Transparency: we make the constraint explicit and consistent.)
# ----------------------------
X_train = clamp01(X_train_raw)

# ----------------------------
# 3) Current best (maximisation)
# ----------------------------
current_best_idx = int(np.argmax(y_train))
current_best_x = X_train[current_best_idx]
current_best_y = float(y_train[current_best_idx])

print("Current best index:", current_best_idx)
print("Current best X (clamped to [0,1]):", current_best_x)
print("Current best y:", current_best_y)

# ----------------------------
# 4) Scaling choice (Transparent)
#    Since the true domain is [0,1]^4, we use fixed bounds:
#       X_min = 0, X_max = 1  => scaled space == raw space.
# ----------------------------
X_min = np.zeros(4, dtype=float)
X_max = np.ones(4, dtype=float)
X_scaled = X_train.copy()  # identical

y_mean = y_train.mean()
y_std = y_train.std() + 1e-12
y_scaled = (y_train - y_mean) / y_std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled, dtype=torch.float32, device=device).unsqueeze(-1)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

# ----------------------------
# 5) MLP surrogate (kept) + robust training
# ----------------------------
class MLP(nn.Module):
    def __init__(self, input_dim=4, hidden=(64, 64), p_dropout=0.1):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h2, 1),
        )

    def forward(self, x):
        return self.net(x)

def train_model(
    model, X, y,
    max_epochs=1400,
    lr=1e-3,
    weight_decay=1e-5,
    patience=80,
    min_delta=1e-4
):
    # Huber for robustness (same intent as Week 9)
    criterion = nn.SmoothL1Loss(beta=1.0)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best = float("inf")
    bad = 0
    model.train()

    for _ in range(max_epochs):
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        lv = float(loss.item())
        if lv < best - min_delta:
            best = lv
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    return best

# ----------------------------
# 6) CV + limited random search (more transparent & reproducible)
#    (We keep tuning, but reduce degrees of freedom for interpretability.)
# ----------------------------
def kfold_indices(n, k=4, seed=123):
    rr = np.random.default_rng(seed)
    idx = np.arange(n)
    rr.shuffle(idx)
    return np.array_split(idx, k)

def cv_mse_for_config(cfg, X_tensor, y_tensor, k=4, seed=123):
    folds = kfold_indices(len(X_tensor), k=k, seed=seed)
    mses = []
    for i in range(k):
        val_idx = folds[i]
        tr_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        X_tr, y_tr = X_tensor[tr_idx], y_tensor[tr_idx]
        X_val, y_val = X_tensor[val_idx], y_tensor[val_idx]

        torch.manual_seed(1000 + i)
        model = MLP(input_dim=4, hidden=cfg["hidden"], p_dropout=cfg["dropout"]).to(device)
        train_model(
            model, X_tr, y_tr,
            max_epochs=cfg["max_epochs"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            patience=cfg["patience"]
        )

        model.eval()
        with torch.no_grad():
            pred = model(X_val)
            mse = nn.MSELoss()(pred, y_val).item()
        mses.append(mse)

    return float(np.mean(mses))

def sample_config(rr):
    # Narrowed search space (transparency + less variance)
    hidden_choices = [(64, 64), (128, 64)]
    dropout_choices = [0.05, 0.10]
    lr_choices = [3e-4, 1e-3]
    wd_choices = [1e-6, 1e-5, 1e-4]
    max_epochs_choices = [1200, 1600]
    patience_choices = [80, 110]
    n_ens_choices = [11]          # fixed for stability
    xi_choices = [0.005, 0.010]   # modest exploration

    return {
        "hidden": hidden_choices[rr.integers(0, len(hidden_choices))],
        "dropout": float(dropout_choices[rr.integers(0, len(dropout_choices))]),
        "lr": float(lr_choices[rr.integers(0, len(lr_choices))]),
        "weight_decay": float(wd_choices[rr.integers(0, len(wd_choices))]),
        "max_epochs": int(max_epochs_choices[rr.integers(0, len(max_epochs_choices))]),
        "patience": int(patience_choices[rr.integers(0, len(patience_choices))]),
        "n_ensemble": int(n_ens_choices[rr.integers(0, len(n_ens_choices))]),
        "xi": float(xi_choices[rr.integers(0, len(xi_choices))]),
    }

N_TRIALS = 18  # fewer trials; diminishing returns + better reporting
K_FOLDS = 4

best_cfg = None
best_cv = float("inf")

print("\n=== Hyperparameter tuning (random search + {}-fold CV) ===".format(K_FOLDS))
for t in range(N_TRIALS):
    cfg = sample_config(rng)
    cv = cv_mse_for_config(cfg, X_tensor_all, y_tensor_all, k=K_FOLDS, seed=123)
    if cv < best_cv:
        best_cv = cv
        best_cfg = cfg
    print(f"trial {t+1:02d}/{N_TRIALS}  cv_mse={cv:.6f}  cfg={cfg}")

print("\n=== Best tuned configuration ===")
print("Best CV-MSE (scaled y):", best_cv)
print("Best cfg:", best_cfg)

# ----------------------------
# 7) Train tuned BOOTSTRAPPED ensemble
# ----------------------------
def train_ensemble(cfg, X_tensor, y_tensor, rr):
    ensemble = []
    n = len(X_tensor)
    for m in range(cfg["n_ensemble"]):
        boot_idx = rr.integers(0, n, size=n)
        Xb = X_tensor[boot_idx]
        yb = y_tensor[boot_idx]

        torch.manual_seed(500 + m)
        model = MLP(input_dim=4, hidden=cfg["hidden"], p_dropout=cfg["dropout"]).to(device)
        train_model(
            model, Xb, yb,
            max_epochs=cfg["max_epochs"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            patience=cfg["patience"]
        )
        ensemble.append(model)
    return ensemble

ensemble = train_ensemble(best_cfg, X_tensor_all, y_tensor_all, rng)

# ----------------------------
# 8) Prediction utilities (original y units) + EI/PI
# ----------------------------
def ensemble_predict(ensemble, X_scaled_tensor):
    preds = []
    with torch.no_grad():
        for model in ensemble:
            model.eval()
            p_scaled = model(X_scaled_tensor).squeeze(-1)
            p_raw = p_scaled * y_std + y_mean
            preds.append(p_raw)
    preds = torch.stack(preds, dim=0)
    return preds.mean(dim=0), preds.std(dim=0) + 1e-9

normal = torch.distributions.Normal(
    torch.tensor(0.0, device=device),
    torch.tensor(1.0, device=device)
)

def expected_improvement(mean, std, best_y, xi=0.01):
    imp = mean - best_y - xi
    Z = imp / std
    ei = imp * normal.cdf(Z) + std * torch.exp(normal.log_prob(Z))
    return torch.where(std > 0, ei, torch.zeros_like(ei))

def probability_of_improvement(mean, std, best_y, xi=0.0):
    imp = mean - best_y - xi
    Z = imp / std
    return normal.cdf(Z)

# ----------------------------
# 9) Week 10 candidate search (same structure, clearer scoring + bounds)
# ----------------------------
DECODING = {
    "temperature": 0.45,  # kept for logging only
    "top_p": 0.80,
    "top_k": 40,
    "max_tokens": 12000,  # candidate budget
}

CANDIDATE_MIX = {
    "local_frac": 0.85,
    "trust_radius": 0.10,
}

REFINE = {
    "topk_seed": 256,
    "per_seed": 24,
    "refine_radius": 0.05
}

SCORE_WEIGHTS = {"w_mean": 0.68, "w_ei": 0.27, "w_pi": 0.05}

# Distance penalty (explicitly framed as "extrapolation risk")
PENALTY = {"lambda_dist": 0.12}

def propose_next_point_week10_greedy_best_by_score(
    ensemble, X_existing_raw01, X_scaled_existing,
    best_x_raw01, rr, xi,
    dup_tol=1e-6,
    decoding=DECODING,
    cand_mix=CANDIDATE_MIX,
    refine=REFINE,
    weights=SCORE_WEIGHTS,
    penalty=PENALTY,
    top_report=10
):
    n_total = int(decoding["max_tokens"])
    local_n = int(cand_mix["local_frac"] * n_total)
    global_n = n_total - local_n

    # scaled==raw for [0,1] domain
    best_scaled = best_x_raw01.astype(np.float32)

    r = float(cand_mix["trust_radius"])
    local = best_scaled + rr.normal(0.0, r, size=(local_n, 4)).astype(np.float32)
    local = clamp01(local)

    global_c = rr.random((global_n, 4), dtype=np.float32)
    coarse_scaled = np.vstack([local, global_c]).astype(np.float32)

    # ----- coarse scoring -----
    X_cand_tensor = torch.tensor(coarse_scaled, dtype=torch.float32, device=device)
    mean, std = ensemble_predict(ensemble, X_cand_tensor)

    best_y = float(np.max(y_train))
    ei = expected_improvement(mean, std, best_y, xi=xi)
    pi = probability_of_improvement(mean, std, best_y, xi=0.0)

    mean_np = mean.detach().cpu().numpy()
    std_np = std.detach().cpu().numpy()
    ei_np = ei.detach().cpu().numpy()
    pi_np = pi.detach().cpu().numpy()

    def norm01(v):
        v = np.asarray(v, dtype=np.float64)
        lo, hi = np.min(v), np.max(v)
        return (v - lo) / (hi - lo + 1e-12)

    mean_n = norm01(mean_np)
    ei_n = norm01(ei_np)
    pi_n = norm01(pi_np)

    # distance penalty in scaled/raw=[0,1] space
    d_near = nearest_dist_scaled(coarse_scaled.astype(np.float64), X_scaled_existing.astype(np.float64))
    d_n = norm01(d_near)

    score_coarse = (
        weights["w_mean"] * mean_n
        + weights["w_ei"] * ei_n
        + weights["w_pi"] * pi_n
        - float(penalty["lambda_dist"]) * d_n
    )

    # top-k seeds
    topk = int(refine["topk_seed"])
    seed_idx = np.argsort(score_coarse)[::-1][:topk]
    seeds = coarse_scaled[seed_idx]

    # ----- refinement -----
    per_seed = int(refine["per_seed"])
    rr_ref = float(refine["refine_radius"])

    refine_points = []
    for s in seeds:
        jitter = s + rr.normal(0.0, rr_ref, size=(per_seed, 4)).astype(np.float32)
        jitter = clamp01(jitter)
        refine_points.append(jitter)

    refine_scaled = np.vstack([coarse_scaled] + refine_points).astype(np.float32)

    X_ref_tensor = torch.tensor(refine_scaled, dtype=torch.float32, device=device)
    mean2, std2 = ensemble_predict(ensemble, X_ref_tensor)

    ei2 = expected_improvement(mean2, std2, best_y, xi=xi)
    pi2 = probability_of_improvement(mean2, std2, best_y, xi=0.0)

    mean2_np = mean2.detach().cpu().numpy()
    std2_np = std2.detach().cpu().numpy()
    ei2_np = ei2.detach().cpu().numpy()
    pi2_np = pi2.detach().cpu().numpy()

    mean2_n = norm01(mean2_np)
    ei2_n = norm01(ei2_np)
    pi2_n = norm01(pi2_np)

    d2_near = nearest_dist_scaled(refine_scaled.astype(np.float64), X_scaled_existing.astype(np.float64))
    d2_n = norm01(d2_near)

    score2 = (
        weights["w_mean"] * mean2_n
        + weights["w_ei"] * ei2_n
        + weights["w_pi"] * pi2_n
        - float(penalty["lambda_dist"]) * d2_n
    )

    # Filter to non-duplicates; greedy best-by-score
    keep = []
    for i in range(len(refine_scaled)):
        x_raw01 = refine_scaled[i]  # already in [0,1]
        if not is_duplicate(x_raw01, X_existing_raw01, tol=dup_tol):
            keep.append(i)

    if len(keep) == 0:
        chosen_i = int(np.argmax(score2))
    else:
        keep = np.array(keep, dtype=int)
        chosen_i = int(keep[np.argmax(score2[keep])])

    def pack(j):
        xr01 = refine_scaled[j].astype(np.float64)
        return {
            "idx": int(j),
            "x_raw01": xr01,
            "mean": float(mean2_np[j]),
            "std": float(std2_np[j]),
            "ei": float(ei2_np[j]),
            "pi": float(pi2_np[j]),
            "score": float(score2[j]),
            "d_near": float(d2_near[j]),
        }

    chosen = pack(chosen_i)

    pool = np.arange(len(refine_scaled)) if len(keep) == 0 else keep
    pool_sorted_score = pool[np.argsort(score2[pool])[::-1]]
    pool_sorted_ei = pool[np.argsort(ei2_np[pool])[::-1]]

    report = {
        "top_by_ei": [pack(j) for j in pool_sorted_ei[:top_report]],
        "top_by_score": [pack(j) for j in pool_sorted_score[:top_report]],
    }

    meta = {
        "best_y": best_y,
        "n_candidates_total": n_total,
        "n_candidates_refined": int(len(refine_scaled)),
        "decoding": decoding,
        "cand_mix": cand_mix,
        "refine": refine,
        "weights": weights,
        "penalty": penalty,
        "selection": "GREEDY_BEST_BY_SCORE",
        "domain_bounds": "[0,1]^4 (enforced)",
        "scaling": "fixed (raw==scaled)",
    }
    return chosen, report, meta

chosen, report, meta = propose_next_point_week10_greedy_best_by_score(
    ensemble=ensemble,
    X_existing_raw01=X_train,       # clamped training set used as "existing points"
    X_scaled_existing=X_scaled,
    best_x_raw01=current_best_x,
    rr=rng,
    xi=best_cfg["xi"],
    dup_tol=1e-6,
    top_report=10
)

next_x = clamp01(chosen["x_raw01"])  # final safeguard, always within [0,1]
next_mean = chosen["mean"]
next_std = chosen["std"]
next_ei = chosen["ei"]
next_pi = chosen["pi"]
next_score = chosen["score"]
next_dnear = chosen["d_near"]

# ----------------------------
# 10) Interpretability add-ons
#     (A) Nearest observed points (context)
#     (B) Local sensitivity via gradient on a single representative model
#         (ensemble mean grad is heavier; we keep it simple & reproducible)
# ----------------------------
nn_idx, nn_dist, nn_y = topk_nearest(next_x, X_train, y_train, k=3)

# Gradient-based local sensitivity (use first model as representative)
rep_model = ensemble[0]
rep_model.eval()
x_t = torch.tensor(next_x[None, :], dtype=torch.float32, device=device, requires_grad=True)
pred_scaled = rep_model(x_t)  # (1,1) in scaled y
pred_raw = pred_scaled * y_std + y_mean
pred_raw.backward()
grad = x_t.grad.detach().cpu().numpy().reshape(-1)

grad_abs = np.abs(grad)
grad_abs_sum = float(grad_abs.sum() + 1e-12)
grad_contrib = grad_abs / grad_abs_sum  # normalized contributions per feature

# ----------------------------
# 11) Report (x_next in 6 decimals)
# ----------------------------
print("\n================ WEEK 10 FUNCTION 4 RESULTS (v5 — TRANSPARENCY + INTERPRETABILITY) ================")
print("Tuned surrogate config:", best_cfg)
print("Best CV-MSE (scaled y):", best_cv)

print("\nCURRENT BEST OBSERVED")
print("x_best =", fmt_x6(current_best_x), ", y_best =", f"{current_best_y:.6f}")

print("\nWEEK 10 SETTINGS (Transparent)")
print("Domain bounds:", meta["domain_bounds"])
print("Scaling:", meta["scaling"])
print("Candidate mix:", meta["cand_mix"])
print("Refinement:", meta["refine"])
print("Decoding (kept for logging):", meta["decoding"])
print("Score weights:", meta["weights"])
print("Extrapolation-risk penalty:", meta["penalty"])
print("Selection:", meta["selection"])
print("xi:", f"{best_cfg['xi']:.6f}")
print("Total coarse candidates:", meta["n_candidates_total"])
print("Total evaluated after refinement:", meta["n_candidates_refined"])

print("\nTOP-10 NON-DUPLICATE CANDIDATES (ranked by EI)")
for i, r in enumerate(report["top_by_ei"], 1):
    print(
        f"{i:02d}) x={fmt_x6(r['x_raw01'])} | mean={r['mean']:.6f} std={r['std']:.6f} "
        f"EI={r['ei']:.6f} PI={r['pi']:.6f} SCORE={r['score']:.6f} d_near={r['d_near']:.6f}"
    )

print("\nTOP-10 NON-DUPLICATE CANDIDATES (ranked by SCORE used for selection)")
for i, r in enumerate(report["top_by_score"], 1):
    print(
        f"{i:02d}) x={fmt_x6(r['x_raw01'])} | mean={r['mean']:.6f} std={r['std']:.6f} "
        f"EI={r['ei']:.6f} PI={r['pi']:.6f} SCORE={r['score']:.6f} d_near={r['d_near']:.6f}"
    )

print("\nRECOMMENDED NEXT POINT (domain-safe; x_next in 6 decimals)")
print("x_next     =", fmt_x6(next_x))
print("mu(x_next) =", f"{next_mean:.6f}")
print("sigma      =", f"{next_std:.6f}")
print("EI         =", f"{next_ei:.6f}")
print("PI         =", f"{next_pi:.6f}")
print("SCORE      =", f"{next_score:.6f}")
print("d_near     =", f"{next_dnear:.6f}", "(distance to nearest observed point in [0,1]^4)")

# ----------------------------
# 12) Week 10 reasoning prompts (explicit + reproducible)
# ----------------------------
delta = next_mean - current_best_y

print("\nWEEK 10 REASONING (Transparency & Interpretability)")
print("Strategy used for this round:")
print("- Exploit locally around current best (85% local sampling) while reserving 15% for global coverage.")
print("- Rank candidates using a transparent weighted score: mean + EI + PI minus a distance penalty (extrapolation risk).")
print("- Refine around the top coarse seeds to increase sample efficiency without inflating candidate budget.")

print("\nHow previous-round patterns influenced this choice:")
print("- Best observed y is near the most recent cluster of good points; the trust radius keeps search near that basin.")
print("- EI/PI still included so we do not purely chase the surrogate mean (guards against overfitting to the surrogate).")

print("\nTransparency / reproducibility notes:")
print("- Domain constraint is enforced explicitly: every proposed x is clamped to [0,1].")
print("- Scaling is fixed (raw==scaled) so another researcher can reproduce without relying on dataset min/max quirks.")
print("- Seeds are fixed (numpy + torch) and all hyperparameters + scoring weights are printed.")

print("\nKey assumption (and how it can limit results):")
print("- Assumption: a local surrogate (MLP ensemble) generalises smoothly within the sampled basin.")
print("  If the true function has sharp discontinuities or a distant, better basin, this local bias can miss it.")

print("\nGaps / potential biases in the dataset:")
print("- Sampling is denser near prior best regions (intentional exploitation), meaning other areas of [0,1]^4 remain underexplored.")
print("- One out-of-bounds historical point existed; we clamp training inputs to maintain consistency with the stated domain.")

print("\nOne significant limitation of this approach:")
print("- Surrogate-driven optimisation can become overconfident where data are sparse; distance penalty mitigates this but cannot remove it.")
print(f"- Predicted Δmean vs best: {delta:.6f} (positive means surrogate expects improvement).")

print("\nINTERPRETABILITY CHECKS")
print("Nearest observed points to x_next (for context):")
for rank, (ii, dd, yy) in enumerate(zip(nn_idx, nn_dist, nn_y), 1):
    print(f"  {rank}) idx={int(ii)}  x={fmt_x6(X_train[int(ii)])}  y={float(yy):.6f}  dist={float(dd):.6f}")

print("\nLocal sensitivity (representative model gradient at x_next; normalized abs contributions):")
for j in range(4):
    print(f"  feature {j+1}: grad={grad[j]: .6e}  contrib={grad_contrib[j]*100:6.2f}%")



Current best index: 37
Current best X (clamped to [0,1]): [0.480706 0.444032 0.354963 0.354729]
Current best y: -0.81371612670717

=== Hyperparameter tuning (random search + 4-fold CV) ===
trial 01/18  cv_mse=0.292924  cfg={'hidden': (64, 64), 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 1e-05, 'max_epochs': 1200, 'patience': 110, 'n_ensemble': 11, 'xi': 0.005}
trial 02/18  cv_mse=0.243593  cfg={'hidden': (128, 64), 'dropout': 0.05, 'lr': 0.0003, 'weight_decay': 1e-05, 'max_epochs': 1600, 'patience': 110, 'n_ensemble': 11, 'xi': 0.01}
trial 03/18  cv_mse=0.274181  cfg={'hidden': (128, 64), 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 1e-06, 'max_epochs': 1600, 'patience': 80, 'n_ensemble': 11, 'xi': 0.01}
trial 04/18  cv_mse=0.264302  cfg={'hidden': (64, 64), 'dropout': 0.05, 'lr': 0.001, 'weight_decay': 0.0001, 'max_epochs': 1600, 'patience': 80, 'n_ensemble': 11, 'xi': 0.01}
trial 05/18  cv_mse=0.244649  cfg={'hidden': (128, 64), 'dropout': 0.05, 'lr': 0.0003, 'weight_decay': 1e-06, 